# Data Preprocessing

- Clean stopped words, punctuation, replace URL template with '__url__' tokens
- Fill NaN/null values with empty string

In [2]:
%pip install nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Importing libraries

In [2]:
import pandas as pd
import re, nltk, string #RegEx
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

In [5]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)

True

## Loading dataset

In [4]:
raw_dataset = pd.read_csv('./data/raw/enron_spam_data.csv')

## Previewing samples in dataset

In [6]:
pd.set_option('display.max_colwidth', None)
raw_dataset[["Subject", "Message"]].head(20)

Subject  \
0                 christmas tree farm pictures   
1                     vastar resources , inc .   
2                 calpine daily gas nomination   
3                                   re : issue   
4                    meter 7268 nov allocation   
5                     mcmullen gas for 11 / 99   
6                        meter 1517 - jan 1999   
7                          duns number changes   
8                                   king ranch   
9                       re : entex transistion   
10                           entex transistion   
11          lst rev dec . 1999 josey ranch nom   
12         2 nd rev dec . 1999 josey ranch nom   
13                        unify close schedule   
14                       meter 1431 - nov 1999   
15                       meter 1431 - nov 1999   
16                           y 2 k - texas log   
17                         re : lyondell citgo   
18   hpl fuel gas buy - back for december 1999   
19  ua 4 - meter 1441 for 11 / 97 - falfurrias   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

## Preprocessing dataset

In [7]:
# Initializing the empty preprocessed dataset first
preprocessed_dataset = pd.DataFrame({'Label': [],
                                     'Subject': [],
                                     'Message': []
                                     })

In [8]:
STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
# Convert labels from NLTK POS Tag to WordNet POS Tag
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [10]:
def preprocessing_message(text: str) -> str:
    # Fill null/NaN values with empty string
    cleaned = '' if pd.isna(text) else str(text)
    
    # Remove Unicode characters
    cleaned = cleaned.encode('ascii', 'ignore').decode('ascii')
    
    # Remove HTML tags
    cleaned = re.sub(r'<[^>]+>', ' ', cleaned)
        
    # Replace https string into token '__url__', with example: "https://example.net" or "example.net"
    cleaned = re.sub(r'^(http[s]?:\\/\\/(www\\.)?|ftp:\\/\\/(www\\.)?|www\\.){1}([0-9A-Za-z-\\.@:%_\+~#=]+)+((\\.[a-zA-Z]{2,3})+)(/(.)*)?(\\?(.)*)?', 
                     '__url__', cleaned)
    
    cleaned = re.sub(r'\b\d+\b', ' ', cleaned)
    
    # Removing punctuation, replace it with space 
    cleaned = re.sub(f"[{re.escape(string.punctuation)}]", ' ', cleaned)
    
    # Removing \n endline, or \t tabline
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    
    tokens = nltk.word_tokenize(cleaned)
    
    pos_tags = nltk.pos_tag(tokens)
    
    final_tokens = []
    
    for word, tag in pos_tags:
        wn_tag = get_wordnet_pos(tag)
        lemma = lemmatizer.lemmatize(word, pos=wn_tag)
        
        lemma_lower = lemma.lower().strip()
        
        if lemma_lower not in STOP_WORDS and len(lemma_lower) > 2:
            final_tokens.append(lemma_lower)
            
    return ' '.join(final_tokens)

In [ ]:
raw_dataset['Subject'] = raw_dataset['Subject'].fillna("")
raw_dataset['Message'] = raw_dataset['Message'].fillna("")

In [13]:
print(f"Number of records before deleting duplicated: {len(raw_dataset)}")

Number of records before deleting duplicated: 33716


In [14]:
# Drop Duplicated Record been detected when performing EDA
raw_dataset = raw_dataset.drop_duplicates(subset=['Subject', 'Message'], keep='first')

In [15]:
print(f"Number of records after dropped duplicated: {len(raw_dataset)}")

Number of records after dropped duplicated: 30494


In [16]:
preprocessed_dataset['Subject'] = raw_dataset['Subject'].apply(preprocessing_message)
preprocessed_dataset['Message'] = raw_dataset['Message'].apply(preprocessing_message)
preprocessed_dataset['Label'] = raw_dataset['Spam/Ham']

In [17]:
# Remove record has null at both Subject and Message column
preprocessed_dataset = preprocessed_dataset[
    (preprocessed_dataset['Subject'] != '') | (preprocessed_dataset['Message'] != '')
]

## Previewing the preprocessed data

In [18]:
preprocessed_dataset.info()

<class 'pandas.DataFrame'>
Index: 30479 entries, 0 to 33714
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Label    30479 non-null  str  
 1   Subject  30479 non-null  str  
 2   Message  30479 non-null  str  
dtypes: str(3)
memory usage: 952.5 KB


In [19]:
preprocessed_dataset[["Subject", "Message"]].head(20)

,Subject,Message
0,christmas tree farm picture,
1,vastar resource inc,gary production high island large block commence saturday gross carlos expect gross tomorrow vastar gross production george forward george weissman hou ect daren farmer carlos rodriguez hou ect ect george weissman hou ect ect melissa graf hou ect ect subject vastar resource inc carlos please call linda get everything set estimate come tomorrow increase follow day base conversation bill fischer bmar forward daren farmer hou ect enron north america corp george weissman daren farmer hou ect ect gary bryan hou ect ect melissa graf hou ect ect subject vastar resource inc darren attached appear nomination vastar resource inc high island large block previously erroneously refer well vastar expect well commence production sometime tomorrow tell linda harris get telephone number gas control provide notification turn tomorrow linda number record voice fax would please see someone contact linda advise submit future nomination via mail fax voice thanks george forward george weissman hou ect linda harris george weissman hou ect ect subject effective mscf min ftp time hour hour hour hour hour hour hour hour hour hour hour hour hour hour hour hour hour
2,calpine daily gas nomination,calpine daily gas nomination doc
3,issue,fyi see note already stella forward stella morris hou ect sherlyn schumack stella morris hou ect ect howard camp hou ect ect subject issue stella already take care yesterday thanks howard camp stella morris hou ect ect sherlyn schumack hou ect ect howard camp hou ect ect stacey neuweiler hou ect ect daren farmer hou ect ect subject issue stella work stacey daren resolve forward howard camp hou ect sherlyn schumack howard camp hou ect ect subject issue create accounting arrangement purchase unocal energy meter deal track volume deal expire
4,meter nov allocation,fyi forward lauri allen hou ect kimberly vaughn lauri allen hou ect ect mary smith hou ect ect subject meter nov allocation lauri put strangas gas get contract daren forward kimberly vaughn hou ect lauri allen kimberly vaughn hou ect ect anita luong hou ect ect howard camp hou ect ect mary smith hou ect ect subject meter nov allocation kim anita volume show allocate reliant contract november nomination reliant point november therefore volume allocate contract please make sure volume move reliant contract prior november close thanks
5,mcmullen gas,jackie since inlet river plant shut last day flow meter mcmullen gas divert meter hpl buy residue gas gas teco vastar vintage tejones swift still see active deal meter path manager teco vastar vintage tejones swift also see gas schedule pop meter please advice need resolve soon possible settlement send payment thanks
6,meter jan,george need follow jan zero receipt package allocate flow deliv package jan zero receipt package zero deliv package buyback incorrectly nominate transport contract ect receipt let know
7,dun number change,fyi forward gary payne hou ect antoine pierre tommy yanowski hou ect ect kathryn bussell hou ect ect gary payne hou ect ect diane niestrath hou ect ect romeo souza hou ect ect michael eiben hou ect ect clem cernosek hou ect ect scotty gilbert hou ect ect dave nommensen hou ect ect david rohan hou ect ect kevin heal cal ect ect richard pinion hou ect ect mary gosnell hou ect ect jason moore hou ect ect samuel schott hou ect ect bernice rodriguez hou ect ect subject dun number change make change wednesday december agree problem dnb number change please notify otherwise make change scheduled dunns number change counterparty number cinergy resource inc energy dynamic management inc south jersey resource group llc transalta energy marketing inc philadelphia gas work thanks rennie
8,king ranch,two field gas difficulty unify system cage ranch since processing agreement accomodates gas king ranch understanding hpl sell liquid king ranch deliver stratton also understanding cent fee deliver gas need method accomodate volume flow hpl meter 

## Export the preprocessed data

In [20]:
preprocessed_dataset.to_csv('./data/preprocessed/enron_spam_data_preprocessed.csv',index=False)